In [1]:
# --- Notebook Test Engine: injected setup (auto-generated) ---
import os as _os
for _k in ('AWS_ACCESS_KEY_ID','AWS_SECRET_ACCESS_KEY','AWS_SESSION_TOKEN','AWS_CREDENTIAL_EXPIRATION'):
    _os.environ.pop(_k, None)
_os.environ.setdefault('AWS_DEFAULT_REGION', 'us-west-2')
_os.environ.setdefault('AWS_REGION', 'us-west-2')
_nte_role = 'arn:aws:iam::674622101542:role/NotebookTestEngine-ProcessingJobRole'
try:
    from sagemaker.core.helper import session_helper as _sh
    _sh.get_execution_role = lambda *a, **k: _nte_role
except Exception:
    pass
try:
    _nte_cf = _os.environ.get('AWS_SHARED_CREDENTIALS_FILE')
    if _nte_cf and _os.path.exists(_nte_cf):
        import configparser as _cfp, datetime as _dt
        import boto3 as _b3, botocore.session as _bcs
        from botocore.credentials import RefreshableCredentials as _RC
        _nte_prof = _os.environ.get('AWS_PROFILE', 'default')
        def _nte_load():
            _cp = _cfp.ConfigParser(); _cp.read(_nte_cf)
            _s = _cp[_nte_prof]
            _tok = _s.get('aws_session_token')
            if not _tok:
                raise RuntimeError('nte: no session token in creds file')
            return {'access_key': _s['aws_access_key_id'],
                    'secret_key': _s['aws_secret_access_key'],
                    'token': _tok,
                    'expiry_time': (_dt.datetime.now(_dt.timezone.utc)
                                    + _dt.timedelta(minutes=9)).isoformat()}
        _nte_creds = _RC.create_from_metadata(metadata=_nte_load(),
                                              refresh_using=_nte_load,
                                              method='nte-shared-file')
        def _nte_get_credentials(self, *a, **k):
            # Honor sessions that were handed explicit credentials
            # (e.g. notebooks that assume_role into other accounts):
            # clobbering them would silently run every cross-account
            # client as the ambient identity. Bare/SDK-created sessions
            # (no explicit creds) still get the refreshable shared-file
            # creds so long-running waits survive credential rotation.
            _ex = getattr(self, '_credentials', None)
            if _ex is not None:
                return _ex
            return _nte_creds
        _bcs.Session.get_credentials = _nte_get_credentials
        _b3.setup_default_session(botocore_session=_bcs.get_session())
        print('nte: refreshable shared-file creds installed')
except Exception as _e:
    print('nte: refreshable-creds shim skipped:', _e)
try:
    from sagemaker.core.resources import ModelPackageGroup as _NteMPG
    _nte_mpg_create = _NteMPG.create
    def _nte_mpg_get_or_create(*_a, **_k):
        try:
            return _nte_mpg_create(*_a, **_k)
        except Exception as _e2:
            if 'already exists' in str(_e2).lower():
                _name = _k.get('model_package_group_name') or (_a[0] if _a else None)
                return _NteMPG.get(_name)
            raise
    _NteMPG.create = staticmethod(_nte_mpg_get_or_create)
    print('nte: ModelPackageGroup.create is now get-or-create')
except Exception as _e:
    print('nte: ModelPackageGroup idempotency shim skipped:', _e)


nte: ModelPackageGroup.create is now get-or-create


# SageMaker V3 JumpStart E2E Training Example

This notebook demonstrates how to use SageMaker V3 to train a JumpStart model from scratch and deploy it for inference.

### Prerequisites
Note: Ensure you have sagemaker and ipywidgets installed in your environment. The ipywidgets package is required to monitor endpoint deployment progress in Jupyter notebooks.


In [2]:
# Import required libraries
import json
import uuid

from sagemaker.serve.model_builder import ModelBuilder
from sagemaker.train.model_trainer import ModelTrainer
from sagemaker.core.jumpstart.configs import JumpStartConfig
from sagemaker.core.resources import EndpointConfig

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Fetched defaults config from location: /opt/ml/processing/input/sm_config.yaml


## Step 1: Configure JumpStart Model for Training

We'll train a HuggingFace Falcon model using JumpStart.

In [3]:
# Configuration
MODEL_ID = "huggingface-spc-bert-base-cased"
MODEL_NAME_PREFIX = "js-e2e-example-model"
ENDPOINT_NAME_PREFIX = "js-e2e-example-endpoint"

# Generate unique identifiers
unique_id = str(uuid.uuid4())[:8]
training_job_name = f"js-training-{unique_id}"
model_name = f"{MODEL_NAME_PREFIX}-{unique_id}"
endpoint_name = f"{ENDPOINT_NAME_PREFIX}-{unique_id}"

print(f"Training job name: {training_job_name}")
print(f"Model name: {model_name}")
print(f"Endpoint name: {endpoint_name}")

Training job name: js-training-c350ccf4
Model name: js-e2e-example-model-c350ccf4
Endpoint name: js-e2e-example-endpoint-c350ccf4


## Step 2: Train the Model

Use ModelTrainer to train the JumpStart model. The training job may take 30+ minutes to complete.

In [4]:
from sagemaker.train.configs import Compute
# Create JumpStart configuration
jumpstart_config = JumpStartConfig(model_id=MODEL_ID)

# Initialize ModelTrainer from JumpStart config
model_trainer = ModelTrainer.from_jumpstart_config(compute=Compute(instance_type="ml.g4dn.xlarge"), 
    jumpstart_config=jumpstart_config, 
    base_job_name=training_job_name, 
    hyperparameters={"epochs": 1}
)

# Train the model
print("Starting model training...")
model_trainer.train()
print(f"Training completed: {training_job_name}")

[07/20/26 07:21:22] INFO     SageMaker session not provided. Using default Session.                  ]8;id=3672135;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=3672136;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py#65\65]8;;\

[07/20/26 07:21:23] INFO     Cannot simulate policies for                                  ]8;id=3672143;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=3672144;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py#363\363]8;;\
                             'arn:aws:iam::674622101542:role/NotebookTestEngine-Processing                         
                             JobRole' (access denied); permission verdict unknown.                                 

                    WARNING  Could not verify permissions for role                         ]8;id=3672150;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=3672151;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py#585\585]8;;\
                             'arn:aws:iam::674622101542:role/NotebookTestEngine-Processing                         
                             JobRole' (caller lacks iam:SimulatePrincipalPolicy).                                  
                             Proceeding with it. If the operation later fails with an                              
                             access-denied error, ensure the role has the required                                 
                             permissions for 'training' (see                                                       
                             IamRoleResolver().get_required_actions('training')) or create                         
                             a dedicated role via                                                                  
                             IamRoleResolver().create_execution_role(role_type='training')                         
                             .                                                                                     

                    INFO     Role not provided. Using validated caller role:                         ]8;id=3672157;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=3672158;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py#90\90]8;;\
                             arn:aws:iam::674622101542:role/NotebookTestEngine-ProcessingJobRole                   

                    INFO     Runs on sagemaker prod, region:us-west-2                                  ]8;id=3672165;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=3672166;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/utils/utils.py#375\375]8;;\

...

                    INFO     js-training-c350ccf4-20260720072125/algo-1-1784532128:              ]8;id=3674307;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=3674308;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py#31585\31585]8;;\
                             Info file not found at                                                                
                             '_input_model_extracted/__models_info__.json'.                                        

                    INFO     js-training-c350ccf4-20260720072125/algo-1-1784532128:              ]8;id=3674313;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=3674314;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py#31585\31585]8;;\
                             ++ echo 'Training Container Execution Completed'                                      

                    INFO     js-training-c350ccf4-20260720072125/algo-1-1784532128:              ]8;id=3674319;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=3674320;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py#31585\31585]8;;\
                             Training Container Execution Completed                                                

[07/20/26 07:56:01] INFO     Final Resource Status: Completed                                    ]8;id=3674326;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=3674327;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py#31591\31591]8;;\

Training completed: js-training-c350ccf4


## Step 3: Build ModelBuilder from Trained Model

Create a ModelBuilder from the training artifacts.

In [5]:
# Build ModelBuilder from trained model
model_builder = ModelBuilder(
    model=model_trainer,
    dependencies={"auto": False}
)

[07/20/26 07:56:02] DEBUG    Auto-detecting optimal instance type for model...           ]8;id=3674334;file:///usr/local/lib/python3.12/dist-packages/sagemaker/serve/model_builder_utils.py\model_builder_utils.py]8;;\:]8;id=3674335;file:///usr/local/lib/python3.12/dist-packages/sagemaker/serve/model_builder_utils.py#341\341]8;;\

                    DEBUG    Using default CPU instance type: ml.m5.large                ]8;id=3674341;file:///usr/local/lib/python3.12/dist-packages/sagemaker/serve/model_builder_utils.py\model_builder_utils.py]8;;\:]8;id=3674342;file:///usr/local/lib/python3.12/dist-packages/sagemaker/serve/model_builder_utils.py#375\375]8;;\

                    INFO     Cannot simulate policies for                                  ]8;id=3674347;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=3674348;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py#363\363]8;;\
                             'arn:aws:iam::674622101542:role/NotebookTestEngine-Processing                         
                             JobRole' (access denied); permission verdict unknown.                                 

[07/20/26 07:56:03] WARNING  Could not verify permissions for role                         ]8;id=3674353;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=3674354;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py#585\585]8;;\
                             'arn:aws:iam::674622101542:role/NotebookTestEngine-Processing                         
                             JobRole' (caller lacks iam:SimulatePrincipalPolicy).                                  
                             Proceeding with it. If the operation later fails with an                              
                             access-denied error, ensure the role has the required                                 
                             permissions for 'serving' (see                                                        
                             IamRoleResolver().get_required_actions('serving')) or create                          
                             a dedicated role via                                                                  
                             IamRoleResolver().create_execution_role(role_type='serving').                         

## Step 4: Build the Model

Build the model artifacts and prepare for deployment.

In [6]:
# Build the model
core_model = model_builder.build(model_name=model_name)
print(f"Model Successfully Created: {core_model.model_name}")

                    INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=3674359;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=3674360;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#287\287]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

[07/20/26 07:56:04] DEBUG    Either inference spec or model is provided. ModelBuilder   ]8;id=3674366;file:///usr/local/lib/python3.12/dist-packages/sagemaker/serve/model_builder_utils.py\model_builder_utils.py]8;;\:]8;id=3674367;file:///usr/local/lib/python3.12/dist-packages/sagemaker/serve/model_builder_utils.py#1380\1380]8;;\
                             is not handling MLflow model input                                                    

Using model 'huggingface-spc-bert-base-cased' with wildcard version identifier '*'. You can pin to version '3.0.11' for more stable results. Note that models may have different input/output signatures after a major version upgrade.


[07/20/26 07:56:05] WARNING  Using model 'huggingface-spc-bert-base-cased' with wildcard version       ]8;id=3674374;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/jumpstart/cache.py\cache.py]8;;\:]8;id=3674375;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/jumpstart/cache.py#624\624]8;;\
                             identifier '*'. You can pin to version '3.0.11' for more stable results.              
                             Note that models may have different input/output signatures after a major             
                             version upgrade.                                                                      

                    DEBUG    JumpStart Model ID detected.                               ]8;id=3674381;file:///usr/local/lib/python3.12/dist-packages/sagemaker/serve/model_builder_utils.py\model_builder_utils.py]8;;\:]8;id=3674382;file:///usr/local/lib/python3.12/dist-packages/sagemaker/serve/model_builder_utils.py#2914\2914]8;;\

...

                    INFO     Cannot simulate policies for                                  ]8;id=3674413;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=3674414;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py#363\363]8;;\
                             'arn:aws:iam::674622101542:role/NotebookTestEngine-Processing                         
                             JobRole' (access denied); permission verdict unknown.                                 

[07/20/26 07:56:07] WARNING  Could not verify permissions for role                         ]8;id=3674419;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=3674420;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py#585\585]8;;\
                             'arn:aws:iam::674622101542:role/NotebookTestEngine-Processing                         
                             JobRole' (caller lacks iam:SimulatePrincipalPolicy).                                  
                             Proceeding with it. If the operation later fails with an                              
                             access-denied error, ensure the role has the required                                 
                             permissions for 'serving' (see                                                        
                             IamRoleResolver().get_required_actions('serving')) or create                          
                             a dedicated role via                                                                  
                             IamRoleResolver().create_execution_role(role_type='serving').                         

[07/20/26 07:56:08] INFO     Creating model with name: js-e2e-example-model-c350ccf4         ]8;id=3674427;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/session_helper.py\session_helper.py]8;;\:]8;id=3674428;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/session_helper.py#1922\1922]8;;\

[07/20/26 07:56:09] INFO     ✅ Model has been created: 'js-e2e-example-model-c350ccf4' using ]8;id=3674434;file:///usr/local/lib/python3.12/dist-packages/sagemaker/serve/model_builder.py\model_builder.py]8;;\:]8;id=3674435;file:///usr/local/lib/python3.12/dist-packages/sagemaker/serve/model_builder.py#3656\3656]8;;\
                             server MMS in SAGEMAKER_ENDPOINT mode (ARN:                                           
                             arn:aws:sagemaker:us-west-2:674622101542:model/js-e2e-example-mo                      
                             del-c350ccf4)                                                                         

Model Successfully Created: js-e2e-example-model-c350ccf4


## Step 5: Deploy the Trained Model

Deploy the trained model to a SageMaker endpoint for real-time inference.

In [7]:
# Deploy the trained model to an endpoint
core_endpoint = model_builder.deploy(endpoint_name=endpoint_name)
print(f"Endpoint Successfully Created: {core_endpoint.endpoint_name}")

                    INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=3674440;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=3674441;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#287\287]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

[07/20/26 07:56:10] INFO     Creating endpoint-config with name                              ]8;id=3674447;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/session_helper.py\session_helper.py]8;;\:]8;id=3674448;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/session_helper.py#1093\1093]8;;\
                             js-e2e-example-endpoint-c350ccf4                                                      

                    INFO     Creating endpoint with name js-e2e-example-endpoint-c350ccf4    ]8;id=3674454;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/session_helper.py\session_helper.py]8;;\:]8;id=3674455;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/session_helper.py#1125\1125]8;;\

[07/20/26 07:56:11] WARNING  Failed to enable live logging: An error occurred                ]8;id=3674461;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/session_helper.py\session_helper.py]8;;\:]8;id=3674462;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/session_helper.py#2844\2844]8;;\
                             (AccessDeniedException) when calling the FilterLogEvents                              
                             operation: User:                                                                      
                             arn:aws:sts::674622101542:assumed-role/NotebookTestEngine-Proce                       
                             ssingJobRole/SageMaker is not authorized to perform:                                  
                             logs:FilterLogEvents on resource:                                                     
                             arn:aws:logs:us-west-2:674622101542:log-group:/aws/sagemaker/En                       
                             dpoints/js-e2e-example-endpoint-c350ccf4:log-stream: because no                       
                             identity-based policy allows the logs:FilterLogEvents action.                         
                             Fallback to default logging...                                                        

[07/20/26 07:59:41] INFO     ✅ Deployment successful: Endpoint                               ]8;id=3674468;file:///usr/local/lib/python3.12/dist-packages/sagemaker/serve/model_builder.py\model_builder.py]8;;\:]8;id=3674469;file:///usr/local/lib/python3.12/dist-packages/sagemaker/serve/model_builder.py#2977\2977]8;;\
                             'js-e2e-example-endpoint-c350ccf4' using MMS in                                       
                             SAGEMAKER_ENDPOINT mode (ARN:                                                         
                             arn:aws:sagemaker:us-west-2:674622101542:endpoint/js-e2e-example                      
                             -endpoint-c350ccf4)                                                                   

Endpoint Successfully Created: js-e2e-example-endpoint-c350ccf4


## Step 6: Test the Endpoint

This endpoint performs text entailment classification, determining the logical relationship between pairs of sentences. The returned scores indicate the model's confidence for different entailment categories (e.g., entailment, contradiction, neutral) - higher scores indicate stronger predictions for each relationship type. 

This sends a test request to the deployed endpoint.

In [8]:
# Test with a sample query for the trained model
test_data = ["The weather is sunny today", "It is not raining"]

result = core_endpoint.invoke(
    body=json.dumps(test_data),
    content_type="application/list-text"
)

entailment_scores = json.loads(result.body.read().decode('utf-8'))
print(f"Result of invoking trained endpoint: {entailment_scores}")


Result of invoking trained endpoint: {'probabilities': [1.2199833393096924, -1.2695157527923584]}


## Step 7: Clean Up Resources

Clean up the created resources to avoid ongoing charges.

In [9]:
# Clean up resources
core_endpoint_config = EndpointConfig.get(endpoint_config_name=core_endpoint.endpoint_name)

# Delete in the correct order
core_model.delete()
core_endpoint.delete()
core_endpoint_config.delete()

print("Trained Model and Endpoint Successfully Deleted!")

[07/20/26 07:59:43] INFO     Deleting Model - js-e2e-example-model-c350ccf4                      ]8;id=3674475;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=3674476;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py#20740\20740]8;;\

                    INFO     Deleting Endpoint - js-e2e-example-endpoint-c350ccf4                ]8;id=3674482;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=3674483;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py#10428\10428]8;;\

                    INFO     Deleting EndpointConfig - js-e2e-example-endpoint-c350ccf4          ]8;id=3674489;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=3674490;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py#11220\11220]8;;\

Trained Model and Endpoint Successfully Deleted!


## Summary

This notebook demonstrated:
1. Training a JumpStart model using ModelTrainer
2. Building a ModelBuilder from training artifacts
3. Building the model for deployment
4. Deploying to a SageMaker endpoint
5. Making inference requests
6. Cleaning up resources

The V3 ModelTrainer and ModelBuilder provide a seamless end-to-end workflow from training to deployment!